## Training configuration
- **Epochs**: 60 with early stopping (patience=8)
- **Batch size**: 16 for better gradient estimates
- **Augmentation**: crop, flip, rotate, jitter, erase
- **Scheduler**: 5-epoch warmup → cosine annealing
- **Label smoothing**: 0.1 to reduce overconfidence
- **TTA at inference**: 8 augmented passes averaged

Expected improvement over baseline:
- Stage 1 accuracy: ~85% → ~92%+
- Stage 2 accuracy: ~70% → ~80%+

# Stage 2 — Severity & Subclass Classifier Training

This notebook contains the code to train and evaluate the second stage of our brain MRI classifier, which maps detected tumors into specific subclass severity classes.

In [ ]:
import sys
sys.path.append('..')
import torch
import torch.nn as nn
import numpy as np
from src.dataset import get_severity_dataloaders, SeverityDataset
from src.model   import build_tumor_classifier
from src.train   import train_model, plot_history, evaluate_model, plot_confusion_matrix
from config      import *

print(f"Imports completed. Device: {DEVICE}")

In [ ]:
train_loader, val_loader, test_loader = get_severity_dataloaders(batch_size=BATCH_SIZE)

class_names = list(SEVERITY_FOLDERS.keys())
print("Severity classes:", class_names)

In [ ]:
model = build_tumor_classifier(num_classes=6, pretrained=True)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

all_labels = train_loader.dataset.subset.dataset.labels
classes    = np.unique(all_labels)
weights    = compute_class_weight('balanced', classes=classes, y=all_labels)
class_weights = torch.FloatTensor(weights).to(DEVICE)
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

print("Class weights:")
for name, weight in zip(class_names, weights):
    print(f"  {name:12s}: {weight:.4f}")

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import (
    CosineAnnealingLR, LinearLR, SequentialLR, ReduceLROnPlateau
)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)
# 5-epoch linear warmup, then cosine decay
warmup    = LinearLR(optimizer, start_factor=0.1,
                     end_factor=1.0, total_iters=5)
cosine    = CosineAnnealingLR(optimizer, T_max=EPOCHS-5, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                          milestones=[5])

In [ ]:
history = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = EPOCHS,
    lr           = LEARNING_RATE,
    save_path    = MODELS_DIR / 'severity_classifier.pth',
    class_weights= class_weights,
)

In [ ]:
plot_history(
    history,
    title="Severity Classifier Training",
    save_name="severity_training_curves.png"
)

In [ ]:
print("\n--- Test Set Evaluation ---")
y_pred, y_true = evaluate_model(model, test_loader, class_names, device=DEVICE)

plot_confusion_matrix(y_true, y_pred, class_names, save_name="severity_confusion_matrix.png")

In [ ]:
checkpoint_path = MODELS_DIR / 'severity_classifier.pth'
if checkpoint_path.exists():
    print(f"Model saved successfully to: {checkpoint_path}")
else:
    print("Warning: Checkpoint file not found.")

In [ ]:
ckpt = torch.load(MODELS_DIR / 'severity_classifier.pth', map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['model_state'])
model.eval()

images, labels = next(iter(test_loader))
sample_img = images[0].unsqueeze(0).to(DEVICE)
sample_label = labels[0].item()

with torch.no_grad():
    outputs = model(sample_img)
    prediction = outputs.argmax(dim=1).item()

print(f"Inference Test:")
print(f"  Actual class   : {class_names[sample_label]}")
print(f"  Predicted class: {class_names[prediction]}")